## Специалист по информационным технологиям

### Агент, отвечающий на вопросы, который является специалистом по информационным технологиям
### Будет использоваться сотрудниками Insurellm, страховой технологической компании
### Агент должен быть точным, а решение должно быть недорогим.

В этом проекте будет использоваться RAG (расширенная генерация результатов поиска), чтобы обеспечить высокую точность работы нашего помощника по вопросам/ответам.

В этой первой реализации будет использоваться простой метод RAG, основанный на грубой силе..

In [1]:
# imports

import os
import glob
from pathlib import Path
from dotenv import load_dotenv
import gradio as gr

In [2]:
# imports for langchain, plotly and Chroma

from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np
import plotly.graph_objects as go
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.embeddings import HuggingFaceEmbeddings

In [ ]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4o-mini"
db_name = "vector_news_db"

In [4]:
# Load environment variables in a file called .env

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

In [5]:
# # Прочитайте документы с помощью загрузчиков LangChain
# # Найдите все из всех вложенных папок нашей базы знаний

# folders = glob.glob(r"c:/news/*")

# def add_metadata(doc, doc_type):
#     doc.metadata["doc_type"] = doc_type
#     return doc

# text_loader_kwargs = {'encoding': 'utf-8'}
# # Если это не сработает, некоторым пользователям Windows может потребоваться раскомментировать следующую строку вместо этого
# # text_loader_kwargs={'autodetect_encoding': True}

# documents = []
# for folder in folders:
#     doc_type = os.path.basename(folder)
#     loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
#     folder_docs = loader.load()
#     documents.extend([add_metadata(doc, doc_type) for doc in folder_docs])

# text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# chunks = text_splitter.split_documents(documents)

# print(f"Total number of chunks: {len(chunks)}")
# print(f"Document types found: {set(doc.metadata['doc_type'] for doc in documents)}")

In [6]:
# Прочитайте документы с помощью загрузчиков LangChain
# Найдите все текстовые файлы в нашей базе знаний (без вложенных папок)

news_dir = Path(r"c:/news/")  # Создаем объект Path для папки news
news_files = list(news_dir.glob("*.md")) # Получаем список объектов Path, представляющих файлы .md в папке news
print(news_files)

def add_metadata(doc, filename):
    """Добавляет метаданные к документу."""
    doc.metadata["filename"] = os.path.splitext(filename)[0]  # Имя файла без расширения
    if "next_bar" in doc.metadata: # Добавляем 'next_bar' только если он уже есть в метаданных
        doc.metadata["next_bar"] = doc.metadata.get("next_bar")  # Безопасное получение, чтобы избежать KeyError
    return doc

text_loader_kwargs = {'encoding': 'utf-8'}
# Если это не сработает, некоторым пользователям Windows может потребоваться раскомментировать следующую строку вместо этого
# text_loader_kwargs={'autodetect_encoding': True}

documents = []
for news_file in news_files:
    try:
        filename = os.path.basename(news_file)  # Получаем имя файла из пути
        loader = TextLoader(news_file, encoding=text_loader_kwargs.get('encoding')) # Используем TextLoader для чтения конкретного файла
        documents.extend(loader.load())  # Добавляем все документы из списка
        document = documents[-1]  # Получаем последний документ (единственный)
        document = add_metadata(document, filename)

    except Exception as e:
        print(f"Ошибка при обработке файла {news_file}: {e}")

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Total number of chunks: {len(chunks)}")
# Выводим только те метаданные, которые у нас есть
if documents:  # Проверяем, что документы были загружены
    print(f"Document filenames: {set(doc.metadata['filename'] for doc in documents)}")
else:
    print("Не удалось найти документы.")

[WindowsPath('c:/news/2025-06-25.md'), WindowsPath('c:/news/2025-06-26.md'), WindowsPath('c:/news/2025-06-27.md'), WindowsPath('c:/news/2025-06-30.md'), WindowsPath('c:/news/2025-07-01.md'), WindowsPath('c:/news/2025-07-02.md'), WindowsPath('c:/news/2025-07-03.md'), WindowsPath('c:/news/2025-07-04.md'), WindowsPath('c:/news/2025-07-07.md'), WindowsPath('c:/news/2025-07-08.md'), WindowsPath('c:/news/2025-07-09.md'), WindowsPath('c:/news/2025-07-10.md'), WindowsPath('c:/news/2025-07-11.md')]
Total number of chunks: 25
Document filenames: {'2025-07-03', '2025-07-07', '2025-07-08', '2025-07-02', '2025-06-25', '2025-06-30', '2025-06-27', '2025-07-10', '2025-07-11', '2025-06-26', '2025-07-09', '2025-07-01', '2025-07-04'}


## Небольшое замечание о встраиваниях и "Фильмах с автоматическим кодированием"

Мы будем отображать каждый фрагмент текста в вектор, который представляет значение текста, что называется встраиванием.

Open air предлагает модель для этого, которую мы будем использовать, вызывая их API с помощью некоторого длинного кода.

Эта модель является примером "LLM с автоматическим кодированием", которая генерирует выходные данные на основе полных входных данных.
Это отличается от всех других конечностей, которые мы обсуждали сегодня, которые известны как "авторегрессивные конечности" и генерируют будущие токены только на основе прошлого контекста.

Другим примером Lms с автоматическим кодированием является BERT от Google. Помимо встраивания, Lms с автоматическим кодированием часто используются для классификации.

### Sidenote

На восьмой неделе мы вернемся к RAG и векторным встраиваниям и будем использовать векторный кодировщик с открытым исходным кодом, чтобы данные никогда не покидали наш компьютер - это важный момент при создании корпоративных систем, и данные должны оставаться внутренними.

In [ ]:
# Поместите фрагменты данных в хранилище векторов, которое связывает векторное вложение с каждым фрагментом
# Chroma - популярная векторная база данных с открытым исходным кодом, основанная на SQLLite

embeddings = OpenAIEmbeddings()

# Если вы предпочитаете использовать свободные векторные вложения из HuggingFace sentence-transformers,
# то замените embeddings = OpenAIEmbeddings()
# на:
# from langchain.embeddings import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Удалить, если уже существует

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

# Создать vectorstore и сохранить его в папке db_name

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Векторное хранилище с {vectorstore._collection.count()} документами")

Vectorstore created with 25 documents


In [8]:
# Давайте исследуем векторы, которые мы создали с помощью Chroma.

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"Есть {count:,} векторов с {dimensions:,} размеры в векторном хранилище")

Есть 25 векторов с 1,536 размеры в векторном хранилище


## Визуализация хранилища векторов

Давайте на минутку взглянем на документы и векторы для их встраивания, чтобы понять, что происходит.

In [9]:
# Предварительная работа (выражаем благодарность Джону Р. за выявление и исправление ошибки в этом!)

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['next_bar'] for metadata in metadatas]
colors = [['green', 'red'][['up', 'down'].index(t)] for t in doc_types]

KeyError: 'next_bar'

In [ ]:
# Нам, людям, проще визуализировать объекты в 2D!
# Уменьшите размерность векторов до 2D, используя t-SNE
# (t-распределенное стохастическое вложение соседей)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

## Пришло время использовать длинную цепочку, чтобы объединить все это воедино.

In [ ]:
# create a new Chat with OpenAI
llm = ChatOpenAI(temperature=0.7, model_name=MODEL)

# Alternative - if you'd like to use Ollama locally, uncomment this line instead
# llm = ChatOpenAI(temperature=0.7, model_name='llama3.2', base_url='http://localhost:11434/v1', api_key='ollama')

# set up the conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# the retriever is an abstraction over the VectorStore that will be used during RAG
retriever = vectorstore.as_retriever()

# putting it together: set up the conversation chain with the GPT 3.5 LLM, the vector store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

In [ ]:
# Let's try a simple question

query = "Пожалуйста, объясните, что такое Insurellm, в нескольких предложениях"
result = conversation_chain.invoke({"question": query})
print(result["answer"])

In [ ]:
# set up a new conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# putting it together: set up the conversation chain with the GPT 4o-mini LLM, the vector store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

## Теперь мы расскажем об этом на радио, используя интерфейс чата -

Быстрый и простой способ создать прототип чата с LLM

In [ ]:
# Wrapping that in a function

def chat(question, history):
    result = conversation_chain.invoke({"question": question})
    return result["answer"]

In [ ]:
# And in Gradio:

view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)

In [ ]:
# Let's investigate what gets sent behind the scenes

from langchain_core.callbacks import StdOutCallbackHandler

llm = ChatOpenAI(temperature=0.7, model_name=MODEL)

memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

retriever = vectorstore.as_retriever()

conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory, callbacks=[StdOutCallbackHandler()])

query = "Кто получил престижную награду IIT award в 2023 году?"
result = conversation_chain.invoke({"question": query})
answer = result["answer"]
print("\nAnswer:", answer)

In [ ]:
# create a new Chat with OpenAI
llm = ChatOpenAI(temperature=0.7, model_name=MODEL)

# set up the conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# the retriever is an abstraction over the VectorStore that will be used during RAG; k is how many chunks to use
retriever = vectorstore.as_retriever(search_kwargs={"k": 25})

# putting it together: set up the conversation chain with the GPT 3.5 LLM, the vector store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

In [ ]:
def chat(question, history):
    result = conversation_chain.invoke({"question": question})
    return result["answer"]

In [ ]:
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)

# Упражнения

Попробуйте применить это к своей собственной папке с данными, чтобы создать персонального специалиста по умственному развитию, эксперта по вашей собственной информации!